The whole purpose of this is to find out why the ddG model does not work so well. 

In [1]:
import pickle
import torch
import numpy as np
import gpytorch
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from rdkit import Chem
from rdkit.Chem import AllChem, rdFingerprintGenerator
from itertools import combinations
from gpytorch.kernels import SpectralMixtureKernel
from rdkit.Chem.Pharm2D import Generate
from rdkit.Chem.Pharm2D import Gobbi_Pharm2D
from rdkit import DataStructs
from sklearn.random_projection import GaussianRandomProjection
from morganKernel import *
from training import *

from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.models.gpytorch import GPyTorchModel
from botorch.optim import optimize_acqf_discrete
from botorch.acquisition import ExpectedImprovement
from botorch.utils.transforms import normalize, unnormalize
from botorch.models.model_list_gp_regression import ModelListGP
import botorch
import os
from botorch.acquisition.monte_carlo import qExpectedImprovement
from botorch.acquisition.multi_objective.monte_carlo import qNoisyExpectedHypervolumeImprovement, qExpectedHypervolumeImprovement
from torch.distributions import Normal
from botorch.sampling import SobolQMCNormalSampler
from botorch.acquisition.multi_objective.objective import MCMultiOutputObjective
import json

device = 'cpu' #torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
with open('data/training_data.pkl', 'rb') as f:
    data_train = pickle.load(f)

with open('data/candidates.pkl', 'rb') as f:
    fingerprint_candidates = pickle.load(f)

with open('data/chem_space.pkl', 'rb') as f:
    chem_space = pickle.load(f)

# with open('data/candidates.pkl', 'rb') as f:
#     fingerprint_candidates = pickle.load(f)

In [3]:
# chem_space['00525']['dG_md']
# get key that corresponds to list from chem_space by comparing the primary
# 'COC-5-gcd': ['COC','COC','COC','COC','[OH]','COC','[OH]','[OH]'], #MD-GP 1 # 01312
# 'CF3-2-bcd':  ['C(F)(F)F','[OH]','[OH]', 'C(F)(F)F','[OH]','[OH]','[OH]'],#MD-GP 1
# 'COC-4-bcd':  ['COC','COC','COC','[OH]','COC','[OH]','[OH]'],#MD-GP 1 
# 'CCl3-2-2-bcd':['C(Cl)(Cl)Cl','[OH]','C(Cl)(Cl)Cl','[OH]','[OH]','[OH]','[OH]',] ,#MD-GP 2
# 'COC-4-2-bcd': ['COC','COC','[OH]','COC','COC','[OH]','[OH]'],#MD-GP 2
# 'CCl-prim-bcd': 'CCl',#MD-GP 2
# 'COC-4-3-bcd': ['COC','COC','COC','COC','[OH]','[OH]','[OH]'],#MD-GP 2
for k in chem_space.keys():
    if chem_space[k]['primary'] == ['COC','[OH]','[OH]','COC','COC','COC','[OH]']:
        print(k)
        

# data_train['delta_vina_dG']

In [23]:
# for k in chem_space.keys():
#     prim=chem_space[k]['primary']
#     if prim.count('COC') == 4 and prim.count('[OH]') == 3:
#         print(k,prim)

In [4]:
choices =  torch.tensor(np.hstack([fingerprint_candidates['X_p'], fingerprint_candidates['X_s']]))
X_all = fingerprint_candidates['X_all'].clone().detach()

X_train = torch.tensor(np.hstack([data_train['X_p'], data_train['X_s']])) 
dG_torch = torch.tensor(data_train['dG'][:,0])
# dG_torch, dG_torch_err, dG_min, dG_max =  normalize_labels(data_train['dG'][:,0], data_train['dG'][:,1])
dG_torch_err = torch.tensor(data_train['dG'][:,1])
# ddG_torch, ddG_torch_err, ddG_min, ddG_max =  normalize_labels(data_train['ddG'][:,0], data_train['ddG'][:,1])

ddG_torch = torch.tensor(data_train['ddG'][:,0])
ddG_torch_err = torch.tensor(data_train['ddG'][:,1])

delta_dG_vina = torch.tensor(data_train['delta_vina_dG'][:,0])
delta_dG_vina_err = torch.tensor(data_train['delta_vina_dG'][:,1])

delta_ddG_vina = torch.tensor(data_train['delta_vina_ddG'][:,0])
delta_ddG_vina_err = torch.tensor(data_train['delta_vina_ddG'][:,1])

train_y = torch.cat([dG_torch.view(-1, 1), ddG_torch.view(-1, 1), delta_dG_vina.view(-1,1), delta_ddG_vina.view(-1,1)], dim=1)
err_y = torch.cat([dG_torch_err.view(-1, 1), ddG_torch_err.view(-1, 1), delta_dG_vina_err.view(-1,1), delta_ddG_vina_err.view(-1,1)], dim=1)

REF_POINT = [-(dG_torch.max() + 5), -(data_train['ddG'][:,0].max() + 5)]
REF_POINT = [0,0]
print("REF_POINT:", REF_POINT)
# REF_POINT = [0,0]

REF_POINT: [0, 0]


In [39]:
# err_y

In [5]:
# loading vina values
vina_dG_means = []
vina_dG_stds = []

vina_ddG_means = []
vina_ddG_stds = []

# keys_sorted = [k for k in chem_space.keys() if chem_space[k].get('ddG_md', None) is None]
for key in chem_space.keys():
    vina_pfos = chem_space[key].get('vina_pfos', [])
    vina_sds = chem_space[key].get('vina_sds', [])

    if len(vina_pfos) > 0 and len(vina_sds) > 0:
        vina_dG_means.append(np.mean(vina_pfos))
        vina_dG_stds.append(np.std(vina_pfos))
        vina_ddG_means.append(np.mean(vina_pfos) - np.mean(vina_sds))
        vina_ddG_stds.append(np.sqrt(np.std(vina_pfos)**2 + np.std(vina_sds)**2))
    else:
        vina_dG_means.append(np.nan)
        vina_dG_stds.append(np.nan)
        vina_ddG_means.append(np.nan)
        vina_ddG_stds.append(np.nan)

vina_dG_means = torch.tensor(vina_dG_means, dtype=torch.float64)
vina_dG_stds = torch.tensor(vina_dG_stds, dtype=torch.float64)
vina_ddG_means = torch.tensor(vina_ddG_means, dtype=torch.float64)
vina_ddG_stds = torch.tensor(vina_ddG_stds, dtype=torch.float64)

# TODO: check if the ordering of vina_means and choices match
# seems to be fine. However, we need the whole vina space and not just the choices. 

In [6]:
class AddVinaMCMultiObjective(MCMultiOutputObjective):
    def __init__(
        self,
        vina_dG_means: torch.Tensor,
        vina_dG_vars: torch.Tensor,
        vina_ddG_means: torch.Tensor,
        vina_ddG_vars: torch.Tensor,
        choices: torch.Tensor,
        x_all: torch.Tensor,
    ):
        """
        Args:
            vina_dG_means: (n_candidates, 1) tensor of Vina dG means
            vina_dG_vars:  (n_candidates, 1) tensor of Vina dG variances
            vina_ddG_means: (n_candidates, 1) tensor of Vina ddG means
            vina_ddG_vars:  (n_candidates, 1) tensor of Vina ddG variances
            choices: (n_candidates, d) tensor of candidate inputs
            x_all: (n_candidates, d) tensor of all candidate inputs
        """
        super().__init__()
        self.vina_dG_means = vina_dG_means
        self.vina_dG_vars = vina_dG_vars
        self.vina_ddG_means = vina_ddG_means
        self.vina_ddG_vars = vina_ddG_vars
        self.choices = choices
        self.x_all = x_all
        self._is_mo = True  # Indicate multi-objective

    @staticmethod
    def find_matching_row(row, mat, tol=1e-6):
        diffs = torch.abs(mat - row)
        matches = torch.all(diffs < tol, dim=1)
        indices = torch.nonzero(matches)
        return indices[0].item() if indices.numel() > 0 else -1

    def forward(self, samples: torch.Tensor, X: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            samples: (sample_shape x batch_shape x q x m) — Δ predictions from GPR
            X: (batch_shape x q x d) or (q x d)

        Returns:
            Adjusted samples: (sample_shape x batch_shape x q x m) representing dG_md and ddG_md
        """
        if X is None:
            raise ValueError("X must be provided.")

        # Add Vina means to the first two objectives of samples, matching by X and self.x_all
        adjusted_samples = samples.clone()
        if len(X.shape) == 3:
            # Batched: (batch_shape, q, d)
            batch_shape, q, d = X.shape
            for b in range(batch_shape):
                for i in range(q):
                    match_idx = self.find_matching_row(X[b, i], self.x_all)
                    adjusted_samples[:, b, i, 0] += self.vina_dG_means[match_idx]
                    adjusted_samples[:, b, i, 1] += self.vina_ddG_means[match_idx]
        else:
            # No batch: (q, d)
            q, d = X.shape
            for i in range(q):
                match_idx = self.find_matching_row(X[i], self.x_all)
                adjusted_samples[:, i, 0] += self.vina_dG_means[match_idx]
                adjusted_samples[:, i, 1] += self.vina_ddG_means[match_idx]

        print('max adjusted_samples:', (-adjusted_samples).max())
        return -adjusted_samples


In [23]:
def initialize_model(train_x, train_y, err_y, dims):
    primary_dim, secondary_dim = dims
    models = [
        AdditiveGPModel_botorch(train_x=train_x, train_y=train_y[:, 0], 
                        likelihood=gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=err_y[:, 0]),
                        primary_dim=primary_dim, secondary_dim=secondary_dim),
        AdditiveGPModel_botorch(train_x=train_x, train_y=train_y[:, 1],
                        likelihood=gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=err_y[:, 1]),
                        primary_dim=primary_dim, secondary_dim=secondary_dim)
    ]
    model = ModelListGP(*models).to(device)
    mll = gpytorch.mlls.SumMarginalLogLikelihood(model.likelihood, model).to(device)
    return model, mll

def fit_gpytorch_model(mll, n_iters=1000, lr=0.1, print_freq=100):
    optimizer = torch.optim.Adam(mll.model.parameters(), lr=lr)
    for i in range(n_iters):
        optimizer.zero_grad()
        output = mll.model(*model.train_inputs)
        loss = -mll(output, mll.model.train_targets)
        loss.backward()
        if (i + 1) % print_freq == 0 or i == 0:
            print('Iter %d/%d - Loss: %.3f   mean_l_p1: %.3f   mean_l_s1: %.3f    mean_l_p2: %.3f   mean_l_s2: %.3f' % (
                i + 1, n_iters, loss.item(),
                mll.model.models[0].covar_module_primary.base_kernel.lengthscale.mean(),
                mll.model.models[0].covar_module_secondary.base_kernel.lengthscale.mean(), # are these just mean values? 
                mll.model.models[1].covar_module_primary.base_kernel.lengthscale.mean(),
                mll.model.models[1].covar_module_secondary.base_kernel.lengthscale.mean(),
            ))
        optimizer.step()
    return mll

    # It is fine to define vina_means_obj and vina_vars_obj on top of the function if they are always derived from the global vina_means and vina_stds tensors and you want to use the same values for every call. 
    # However, if you want opt_qnehvi_get_obs to be more flexible and reusable with different Vina statistics, you should pass them as arguments.
    # For your current workflow, defining them on top is acceptable, but for modularity and testing, passing as arguments is better.
def opt_qnehvi_get_obs(model, train_x, choices):
    # Prepare Vina means and variances for all choices
    # vina_means_obj = vina_means.view(-1, 1)
    # vina_vars_obj = (vina_stds ** 2).view(-1, 1)
    vina_dG_means_obj = vina_dG_means.view(-1, 1)
    vina_dG_vars_obj = (vina_dG_stds ** 2).view(-1, 1)
    vina_ddG_means_obj = vina_ddG_means.view(-1, 1)
    vina_ddG_vars_obj = (vina_ddG_stds ** 2).view(-1, 1)

    # Instantiate the custom multi-objective
    objective = AddVinaMCMultiObjective(
        vina_dG_means=vina_dG_means_obj,
        vina_dG_vars=vina_dG_vars_obj,
        vina_ddG_means=vina_ddG_means_obj,
        vina_ddG_vars=vina_ddG_vars_obj,
        choices=choices,
        x_all=X_all
    )

    print('_______________________hhhsh__________________________')
    print('objective:', objective)

    sampler = SobolQMCNormalSampler(sample_shape=torch.Size([1028]))
    acq_func = qNoisyExpectedHypervolumeImprovement(
        model=model,
        ref_point=REF_POINT,
        X_baseline=train_x,
        prune_baseline=False,
        sampler=sampler,
        objective=objective,
    )

    print('_______________________hhhsh__________________________')

    # optimize
    candidates, _ = optimize_acqf_discrete(
        acq_function=acq_func,
        q=5,
        choices=choices,
        max_batch_size=50,
        unique=True
    )
    new_x = candidates.detach()
    # print('Hi----',new_x)
    # with torch.no_grad(), gpytorch.settings.fast_pred_var():
        # posterior = self.model.posterior(new_x)
        # predictions = model.likelihood(model(new_x))
    # predictions = model.likelihood(model(new_x))
    new_post = model.posterior(new_x)
    # new_post = model.likelihood(model(*new_x.unsqueeze(0)))
    new_obj = new_post.mean.detach()
    # new_obj = predictions.mean
    new_obj_err = (new_post.variance).detach()
    # new_obj_err = (predictions.variance/2).sqrt()
    return new_x, new_obj, new_obj_err

In [8]:
# ?optimize_acqf_discrete
# train_y[:,2:], err_y[:,2:]
# choices

In [24]:
pca_components = 20 # change this in the source code as needed
model, mll = initialize_model(X_train, train_y[:,2:], err_y[:,2:], dims=[pca_components, pca_components]) # use variance - maybe try (2*std)**2
# we pass just regular delta values instead of negative. -(delta + vina) must come out from the final objective

In [25]:
mll.train()
model.train()
mll = fit_gpytorch_model(mll, n_iters=3000)
# mll = fit_gpytorch_mll(mll)

Iter 1/3000 - Loss: 22.952   mean_l_p1: 5.000   mean_l_s1: 5.000    mean_l_p2: 5.000   mean_l_s2: 5.000
Iter 100/3000 - Loss: 8.614   mean_l_p1: 2.700   mean_l_s1: 4.665    mean_l_p2: 6.425   mean_l_s2: 4.234
Iter 200/3000 - Loss: 7.801   mean_l_p1: 3.833   mean_l_s1: 4.821    mean_l_p2: 7.773   mean_l_s2: 4.671
Iter 300/3000 - Loss: 5.189   mean_l_p1: 4.637   mean_l_s1: 4.865    mean_l_p2: 8.952   mean_l_s2: 4.883
Iter 400/3000 - Loss: 4.893   mean_l_p1: 4.638   mean_l_s1: 4.673    mean_l_p2: 9.893   mean_l_s2: 5.043
Iter 500/3000 - Loss: 4.700   mean_l_p1: 4.638   mean_l_s1: 4.503    mean_l_p2: 10.708   mean_l_s2: 5.179
Iter 600/3000 - Loss: 4.560   mean_l_p1: 4.638   mean_l_s1: 4.524    mean_l_p2: 11.441   mean_l_s2: 5.301
Iter 700/3000 - Loss: 4.461   mean_l_p1: 4.638   mean_l_s1: 4.559    mean_l_p2: 12.112   mean_l_s2: 5.414
Iter 800/3000 - Loss: 4.382   mean_l_p1: 4.638   mean_l_s1: 4.578    mean_l_p2: 12.740   mean_l_s2: 5.520
Iter 900/3000 - Loss: 4.317   mean_l_p1: 4.638   mea

In [46]:
# X_train.shape, choices.shape, vina_means.view(-1, 1).shape, (vina_stds ** 2).view(-1, 1).shape

In [26]:
mll.eval()
model.eval()
outputs = []
new_x, new_obj, new_obj_err = opt_qnehvi_get_obs(model, X_train, choices)

_______________________hhhsh__________________________
objective: AddVinaMCMultiObjective()
max adjusted_samples: tensor(63.2737, dtype=torch.float64)
_______________________hhhsh__________________________
max adjusted_samples: tensor(68.5204, dtype=torch.float64)
max adjusted_samples: tensor(49.9186, dtype=torch.float64)
max adjusted_samples: tensor(51.2830, dtype=torch.float64)
max adjusted_samples: tensor(48.1667, dtype=torch.float64)
max adjusted_samples: tensor(49.0504, dtype=torch.float64)
max adjusted_samples: tensor(49.8700, dtype=torch.float64)
max adjusted_samples: tensor(49.7290, dtype=torch.float64)
max adjusted_samples: tensor(69.3812, dtype=torch.float64)
max adjusted_samples: tensor(67.6314, dtype=torch.float64)
max adjusted_samples: tensor(70.3778, dtype=torch.float64)
max adjusted_samples: tensor(65.4001, dtype=torch.float64)
max adjusted_samples: tensor(67.9118, dtype=torch.float64)
max adjusted_samples: tensor(66.7934, dtype=torch.float64)
max adjusted_samples: tenso

In [27]:
new_obj_err # these are the model posteriors. need to be adjusted with vina values
new_obj

tensor([[-12.5180,  -8.7407],
        [-12.5180,  -2.3329],
        [-12.5180,  -4.0039],
        [-12.5180,  -1.9838],
        [-12.5180,  -3.5056]], dtype=torch.float64)

In [28]:
# new_obj_err


# find_matching_row(fingerprint_candidates['X_all'][234],fingerprint_candidates['X_all'])

In [29]:
# define the iteration number for BO

iteration = 5
output_dir = f'iteration-{str(iteration).zfill(3)}-delta-learning'

# just to find the corresponding geometry
def find_matching_row(mat, row, tol=1e-6):
    diffs = torch.abs(mat - row)
    matches = torch.all(diffs < tol, dim=1)
    indices = torch.nonzero(matches)
    return indices[0].item() if indices.numel() > 0 else -1

next_iteration = {}
for cand_index in range(5): # for q = 5
    os.makedirs(f'{output_dir}/', exist_ok=True)
    next_arg = find_matching_row(new_x[cand_index], fingerprint_candidates['X_all'])
    index = f'{next_arg:05d}'
    next_sample = chem_space[index]
    vina_pfos_i = chem_space[index]['vina_pfos']
    vina_sds_i = chem_space[index]['vina_sds']
    dG_vina_i = np.mean(vina_pfos_i)
    dG_vina_i_std = np.std(vina_pfos_i)
    ddG_vina_i = dG_vina_i - np.mean(vina_sds_i)
    ddG_vina_i_std = np.sqrt(dG_vina_i_std**2 + np.std(vina_sds_i)**2)
    print(index)
    pred_dG = [new_obj[cand_index, 0].item() + dG_vina_i, np.sqrt(new_obj_err[cand_index, 0].item() + dG_vina_i_std**2)]
    pred_ddG = [new_obj[cand_index, 1].item() + ddG_vina_i, np.sqrt(new_obj_err[cand_index, 1].item() + ddG_vina_i_std**2)]
    # print('dG: ',new_obj[cand_index, 0].item() + dG_vina_i, np.sqrt(new_obj_err[cand_index, 0].item() + dG_vina_i_std**2))
    # print('ddG: ',new_obj[cand_index, 1].item() + ddG_vina_i, np.sqrt(new_obj_err[cand_index, 1].item() + ddG_vina_i_std**2))

    

    next_iteration[index] = next_sample
    next_iteration[index]['delta_dG_gpr'] = [new_obj[cand_index, 0].item(), new_obj_err[cand_index, 0].item()]
    next_iteration[index]['delta_ddG_gpr'] = [new_obj[cand_index, 1].item(), new_obj_err[cand_index, 1].item()]
    next_iteration[index]['pred_dG'] = pred_dG
    next_iteration[index]['pred_ddG'] = pred_ddG

# save next iteration as dict
with open(f'{output_dir}/candidates.json', 'wb') as f:
    pickle.dump(next_iteration, f)

# Save next_iteration as JSON, skipping keys with numpy arrays
def to_json_serializable(d):
    """Recursively convert dict to JSON-serializable, skipping numpy arrays for certain keys."""
    result = {}
    for k, v in d.items():
        if k in ['fingerprint_primary', 'fingerprint_secondary']:
            continue  # skip these keys
        elif isinstance(v, dict):
            result[k] = to_json_serializable(v)
        elif isinstance(v, (list, tuple)):
            # Recursively process lists/tuples
            result[k] = [
                to_json_serializable(i) if isinstance(i, dict) else i for i in v
            ]
        else:
            result[k] = v
    return result

json_ready = {}
for key, value in next_iteration.items():
    json_ready[key] = to_json_serializable(value)

with open(f"{output_dir}/next_iteration.json", "w") as json_file:
    json.dump(json_ready, json_file, indent=4)

00841
00072
00485
00514
00784


In [30]:
# next_iteration['00464'].keys()
# # 'primary', 'secondary', 
# 'CD', 'primary', 'secondary', 'dG_md', 'ddG_md', 'autodock', 'delta_dG', 'delta_ddG', 'vina_pfos', 'vina_sds', 'delta_dG_gpr', 'delta_ddG_gpr', 'pred_dG', 'pred_ddG'

In [ ]:
# next_iteration
# f'{next_arg:05d}'
for next_arg in range(5):
    i = list(next_iteration.keys())[next_arg]
    dG_predicted = np.mean(chem_space[i]['vina_pfos']) + next_iteration[i]['delta_dG_gpr'][0]
    ddG_predicted = np.mean(chem_space[i]['vina_pfos']) - np.mean(chem_space[i]['vina_sds']) + next_iteration[i]['delta_ddG_gpr'][0]
    print(f'{i}: dG: {dG_predicted:1.2f}, ddG: {ddG_predicted:1.2f}')
    # print(f'{np.mean(chem_space[i]['
    # print(f'{:1.2f}, {np.mean(chem_space[i]['vina_pfos']) - np.mean(chem_space[i]['vina_sds']) + next_iteration[i]['delta_ddG_gpr'][0]:1.2f}')
# next_iteration
# next_iteration[i]['delta_ddG_gpr'][0]
# next_iteration[f'{next_arg:05d}']['delta_dG_gpr']

00841: dG: -39.42, ddG: -19.91
00072: dG: -43.60, ddG: -15.16
00485: dG: -41.57, ddG: -14.86
00514: dG: -41.74, ddG: -14.03
00784: dG: -39.82, ddG: -14.44


IndexError: list index out of range

In [32]:
next_iteration.keys()
for key in next_iteration.keys():
    print(key, next_iteration[key]['primary'])

00841 ['[N+](=O)[O-]', '[N+](=O)[O-]', '[OH]', '[N+](=O)[O-]', '[OH]', '[N+](=O)[O-]', '[OH]']
00072 ['COCOC', 'COCOC', 'COCOC', 'COCOC', 'COCOC', 'COCOC', 'COCOC']
00485 ['C(Cl)(Cl)C(Cl)(Cl)C(Cl)(Cl)Cl', 'C(Cl)(Cl)C(Cl)(Cl)C(Cl)(Cl)Cl', 'C(Cl)(Cl)C(Cl)(Cl)C(Cl)(Cl)Cl', '[OH]', '[OH]', '[OH]', '[OH]']
00514 ['C(F)(F)C(F)(F)C(F)(F)F', 'C(F)(F)C(F)(F)C(F)(F)F', '[OH]', '[OH]', '[OH]', '[OH]', '[OH]']
00784 ['OC(F)(F)C(F)(F)F', '[OH]', 'OC(F)(F)C(F)(F)F', '[OH]', '[OH]', '[OH]', '[OH]']


In [33]:
next_iteration.keys()

dict_keys(['00841', '00072', '00485', '00514', '00784'])

### Single objective training Just for testing purposes

In [ ]:
n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store errors
rmse_list = []
pca_components=20

# Create subplots
n_cols = 3
n_rows = math.ceil(n_splits / n_cols)
fig, axs = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
fig.suptitle("Train + Test Parity Plots for GPR Cross-Validation", fontsize=16)
axs = axs.flatten()

for fold, (train_index, test_index) in enumerate(kf.split(X_train)):
    X_train_kf = X_train[train_index]
    X_test = X_train[test_index]
    y_train = train_y[:,1][train_index]
    y_test = train_y[:,1][test_index]
    y_train_std = err_y[:,1][train_index]


    likelihood = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=2*np.sqrt(y_train_std))
    model = AdditiveGPModel_gpytorch(X_train_kf, y_train, likelihood, pca_components, pca_components)
    # model = MultiplicativeGPModel(train_x, train_y, likelihood, pca_components, pca_components)
    
    # Marginal log likelihood for training
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    
    # Train model
    model = train_model(model, mll, X_train_kf, y_train)
    test_noise = torch.full_like(y_test, y_train_std.mean())
    
    # Make predictions
    model.eval()
    likelihood.eval()
    
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_train = likelihood(model(X_train_kf.clone()))
        pred_test = likelihood(model(X_test), noise=test_noise)
        
        y_pred_train = pred_train.mean.numpy()
        y_std_train = pred_train.variance.sqrt().numpy()
        y_pred_test = pred_test.mean.numpy()
        y_std_test = pred_test.variance.sqrt().numpy()
    
    # Calculate metrics
    rmse_train = mean_squared_error(y_train, y_pred_train, squared=False)
    rmse = mean_squared_error(y_test, y_pred_test, squared=False)
    r2 = r2_score(y_test, y_pred_test)
    rmse_list.append(rmse)
    print(f"Fold {fold+1}: RMSE = {rmse:.4f}, R2 = {r2:.4f}")
    
    # Plot parity
    ax = axs[fold]
    ax.set_aspect('equal', adjustable='box')
    # ax.scatter(y_train, y_pred_train, color='dodgerblue', alpha=0.7, label='Train')
    # ax.scatter(y_test, y_pred_test, color='firebrick', alpha=0.7, label='Test')
    # Plot training predictions with error bars
    ax.errorbar(
        y_train, y_pred_train, yerr=y_std_train,
        fmt='o', color='dodgerblue', alpha=0.6, label='Train', markersize=5, capsize=2
    )
    
    # Plot test predictions with error bars
    ax.errorbar(
        y_test, y_pred_test, yerr=y_std_test,
        fmt='o', color='firebrick', alpha=0.6, label='Test', markersize=5, capsize=2
    )

    # min_y = min(train_y[:,1].min(), y_test.min()) - 10
    # max_y = max(train_y[:,1].max(), y_test.max()) + 10
    # ax.plot([min_y, max_y], [min_y, max_y], 'k--', lw=1)
    # ax.set_xlim(min_y, max_y)
    # ax.set_ylim(min_y, max_y)
    ax.set_xlabel("Actual ΔΔG")
    ax.set_ylabel("Predicted ΔΔG")
    ax.set_title(f"Fold {fold+1}\nTrain RMSE={rmse_train:.2f}, Test RMSE={rmse:.2f}")
    ax.legend()
    ax.grid(False)

In [ ]:
n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store errors
rmse_list = []
pca_components=20

# Create subplots
n_cols = 3
n_rows = math.ceil(n_splits / n_cols)
fig, axs = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
fig.suptitle("Train + Test Parity Plots for GPR Cross-Validation", fontsize=16)
axs = axs.flatten()

for fold, (train_index, test_index) in enumerate(kf.split(X_train)):
    X_train_kf = X_train[train_index]
    X_test = X_train[test_index]
    y_train = train_y[:,0][train_index]
    y_test = train_y[:,0][test_index]
    y_train_std = err_y[:,0][train_index]


    likelihood = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=2*np.sqrt(y_train_std))
    model = AdditiveGPModel_gpytorch(X_train_kf, y_train, likelihood, pca_components, pca_components)
    # model = MultiplicativeGPModel(train_x, train_y, likelihood, pca_components, pca_components)
    
    # Marginal log likelihood for training
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    
    # Train model
    model = train_model(model, mll, X_train_kf, y_train)
    test_noise = torch.full_like(y_test, y_train_std.mean())
    
    # Make predictions
    model.eval()
    likelihood.eval()
    
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred_train = likelihood(model(X_train_kf.clone()))
        pred_test = likelihood(model(X_test), noise=test_noise)
        
        y_pred_train = pred_train.mean.numpy()
        y_std_train = pred_train.variance.sqrt().numpy()
        y_pred_test = pred_test.mean.numpy()
        y_std_test = pred_test.variance.sqrt().numpy()
    
    # Calculate metrics
    rmse_train = mean_squared_error(y_train, y_pred_train, squared=False)
    rmse = mean_squared_error(y_test, y_pred_test, squared=False)
    r2 = r2_score(y_test, y_pred_test)
    rmse_list.append(rmse)
    print(f"Fold {fold+1}: RMSE = {rmse:.4f}, R2 = {r2:.4f}")
    
    # Plot parity
    ax = axs[fold]
    ax.set_aspect('equal', adjustable='box')
    # ax.scatter(y_train, y_pred_train, color='dodgerblue', alpha=0.7, label='Train')
    # ax.scatter(y_test, y_pred_test, color='firebrick', alpha=0.7, label='Test')
    # Plot training predictions with error bars
    ax.errorbar(
        y_train, y_pred_train, yerr=y_std_train,
        fmt='o', color='dodgerblue', alpha=0.6, label='Train', markersize=5, capsize=2
    )
    
    # Plot test predictions with error bars
    ax.errorbar(
        y_test, y_pred_test, yerr=y_std_test,
        fmt='o', color='firebrick', alpha=0.6, label='Test', markersize=5, capsize=2
    )

    min_y = min(train_y[:,0].min(), y_test.min()) - 5
    max_y = max(train_y[:,0].max(), y_test.max()) + 5
    ax.plot([min_y, max_y], [min_y, max_y], 'k--', lw=1)
    ax.set_xlim(min_y, max_y)
    ax.set_ylim(min_y, max_y)
    ax.set_xlabel("Actual ΔΔG")
    ax.set_ylabel("Predicted ΔΔG")
    ax.set_title(f"Fold {fold+1}\nTrain RMSE={rmse_train:.2f}, Test RMSE={rmse:.2f}")
    ax.legend()
    ax.grid(False)